In [1]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm


In [2]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)


device: mps


In [3]:
model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
model.eval()

yes_ids = tokenizer.encode("yes", add_special_tokens=False)
no_ids = tokenizer.encode("no", add_special_tokens=False)

print("model:", model_name)
print("decoder_start_token_id:", model.config.decoder_start_token_id)
print("yes ids:", yes_ids, "decoded:", tokenizer.decode(yes_ids))
print("no ids:", no_ids, "decoded:", tokenizer.decode(no_ids))

assert len(yes_ids) == 1, f"Expected single token for 'yes', got {yes_ids}"
assert len(no_ids) == 1, f"Expected single token for 'no', got {no_ids}"

yes_token_id = yes_ids[0]
no_token_id = no_ids[0]
start_token_id = model.config.decoder_start_token_id
if start_token_id is None:
    start_token_id = tokenizer.pad_token_id

print("yes_token_id:", yes_token_id)
print("no_token_id:", no_token_id)
print("start_token_id:", start_token_id)


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


model: google/flan-t5-small
decoder_start_token_id: 0
yes ids: [4273] decoded: yes
no ids: [150] decoded: no
yes_token_id: 4273
no_token_id: 150
start_token_id: 0


In [4]:
ds = load_dataset("glue", "mrpc", split="validation")
print(ds)
print(ds[0])

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])

prompts = [
    f"Do these two sentences have the same meaning? Answer yes or no.\nSentence 1: {a}\nSentence 2: {b}"
    for a, b in zip(sent1, sent2)
]

print("num_examples:", len(y_true))
print("positive_rate:", y_true.mean())
print("sample_prompt:\n", prompts[0])


Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647
sample_prompt:
 Do these two sentences have the same meaning? Answer yes or no.
Sentence 1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
Sentence 2: " The foodservice pie business does not fit our long-term growth strategy .


In [5]:
batch_size = 32
preds = []
yes_logits_all = []
no_logits_all = []

with torch.no_grad():
    for i in tqdm(range(0, len(prompts), batch_size)):
        batch_prompts = prompts[i:i + batch_size]

        enc = tokenizer(
            batch_prompts,
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt",
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        decoder_input_ids = torch.full(
            (len(batch_prompts), 1),
            start_token_id,
            dtype=torch.long,
            device=device,
        )

        outputs = model(**enc, decoder_input_ids=decoder_input_ids)
        first_step_logits = outputs.logits[:, 0, :]

        yes_logits = first_step_logits[:, yes_token_id]
        no_logits = first_step_logits[:, no_token_id]

        batch_preds = (yes_logits > no_logits).long().cpu().numpy()
        preds.extend(batch_preds.tolist())
        yes_logits_all.extend(yes_logits.detach().cpu().numpy().tolist())
        no_logits_all.extend(no_logits.detach().cpu().numpy().tolist())

y_pred = np.array(preds)
yes_logits_all = np.array(yes_logits_all)
no_logits_all = np.array(no_logits_all)
print("done")


  0%|          | 0/13 [00:00<?, ?it/s]

done


In [6]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"], zero_division=0))


{'accuracy': 0.6838235294117647, 'f1': 0.7393939393939394}
                precision    recall  f1-score   support

not_paraphrase       0.50      0.74      0.60       129
    paraphrase       0.85      0.66      0.74       279

      accuracy                           0.68       408
     macro avg       0.67      0.70      0.67       408
  weighted avg       0.74      0.68      0.69       408



In [7]:
for i in range(5):
    print("=" * 80)
    print("idx:", i)
    print("prompt:\n", prompts[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]), "label:", "yes" if int(y_pred[i]) == 1 else "no")
    print("yes_logit:", float(yes_logits_all[i]))
    print("no_logit:", float(no_logits_all[i]))


idx: 0
prompt:
 Do these two sentences have the same meaning? Answer yes or no.
Sentence 1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
Sentence 2: " The foodservice pie business does not fit our long-term growth strategy .
true: 1 pred: 1 label: yes
yes_logit: -1.8457680940628052
no_logit: -2.246473550796509
idx: 1
prompt:
 Do these two sentences have the same meaning? Answer yes or no.
Sentence 1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
Sentence 2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
true: 0 pred: 0 label: no
yes_logit: -4.14354133605957
no_logit: -2.2493021488189697
idx: 2
prompt:
 Do these two sentences have the same meaning? Answer yes or no.
Sentence 1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
Sent

In [8]:
mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("prompt:\n", prompts[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))
    print("yes_logit:", float(yes_logits_all[i]))
    print("no_logit:", float(no_logits_all[i]))


num_errors: 129
idx: 3
prompt:
 Do these two sentences have the same meaning? Answer yes or no.
Sentence 1: The AFL-CIO is waiting until October to decide if it will endorse a candidate .
Sentence 2: The AFL-CIO announced Wednesday that it will decide in October whether to endorse a candidate before the primaries .
true: 1 pred: 0
yes_logit: -3.1410040855407715
no_logit: -2.7465322017669678
idx: 7
prompt:
 Do these two sentences have the same meaning? Answer yes or no.
Sentence 1: This integrates with Rational PurifyPlus and allows developers to work in supported versions of Java , Visual C # and Visual Basic .NET.
Sentence 2: IBM said the Rational products were also integrated with Rational PurifyPlus , which allows developers to work in Java , Visual C # and VisualBasic .Net.
true: 1 pred: 0
yes_logit: -2.598281145095825
no_logit: -2.3498666286468506
idx: 9
prompt:
 Do these two sentences have the same meaning? Answer yes or no.
Sentence 1: The results appear in the January issue of 

In [9]:
summary = {
    "dataset": "glue/mrpc",
    "split": "validation",
    "model": model_name,
    "device": str(device),
    "num_examples": len(ds),
    "accuracy": float(acc),
    "f1": float(f1),
}
summary


{'dataset': 'glue/mrpc',
 'split': 'validation',
 'model': 'google/flan-t5-small',
 'device': 'mps',
 'num_examples': 408,
 'accuracy': 0.6838235294117647,
 'f1': 0.7393939393939394}